# Threshold calibration Drive workflow

该 notebook 用于在 Colab 中从 Google Drive 前序结果包生成 threshold calibration 结果包, 并把 zip 保存回 Google Drive。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
paper_run_name = os.environ.setdefault('SLM_WM_PAPER_RUN_NAME', os.environ.get('SLM_WM_PAPER_RUN_NAME', 'pilot_paper'))
prompt_file_by_run = {
    'pilot_paper': 'configs/paper_main_pilot_paper_prompts.txt',
    'full_paper': 'configs/paper_main_full_paper_prompts.txt',
}
drive_result_root = os.environ.setdefault(
    'SLM_WM_DRIVE_RESULT_ROOT',
    f'/content/drive/MyDrive/SLM/{paper_run_name}_results',
)
paper_run_sample_count = os.environ.setdefault('SLM_WM_PAPER_RUN_SAMPLE_COUNT', 'all')
prompt_count_by_run = {'pilot_paper': 600, 'full_paper': 6000}
def resolve_paper_run_count(value):
    normalized = str(value).strip().lower()
    if normalized in {'', 'all', 'none', 'unlimited'}:
        return int(prompt_count_by_run.get(paper_run_name, prompt_count_by_run['pilot_paper']))
    resolved = int(normalized)
    return int(prompt_count_by_run.get(paper_run_name, prompt_count_by_run['pilot_paper'])) if resolved <= 0 else resolved
paper_run_expected_sample_count = resolve_paper_run_count(paper_run_sample_count)
paper_run_minimum_clean_negative_count = os.environ.setdefault('SLM_WM_PAPER_RUN_MINIMUM_CLEAN_NEGATIVE_COUNT', '100')
paper_run_dataset_minimum_count = os.environ.setdefault('SLM_WM_PAPER_RUN_DATASET_QUALITY_MINIMUM_COUNT', '100')
os.environ.setdefault('SLM_WM_PROTOCOL_PROFILE', f'{paper_run_name}_fixed_fpr_0_01')
os.environ.setdefault('SLM_WM_PROMPT_SET', paper_run_name)
os.environ.setdefault('SLM_WM_PROMPT_FILE', prompt_file_by_run.get(paper_run_name, prompt_file_by_run['pilot_paper']))
os.environ.setdefault('SLM_WM_THRESHOLD_CALIBRATION_DRIVE_DIR', f'{drive_result_root}/threshold_calibration')
os.environ.setdefault('SLM_WM_ATTENTION_INJECTION_DRIVE_DIR', f'{drive_result_root}/attention_latent_injection')
os.environ.setdefault('SLM_WM_ALIGNED_RESCORING_DRIVE_DIR', f'{drive_result_root}/aligned_rescoring')
os.environ.setdefault('SLM_WM_THRESHOLD_TARGET_FPR', '0.01')
os.environ.setdefault('SLM_WM_THRESHOLD_MAX_CONTENT_RECORDS', 'all')
os.environ.setdefault('SLM_WM_THRESHOLD_MINIMUM_CLEAN_NEGATIVE_COUNT', paper_run_minimum_clean_negative_count)
os.environ.setdefault('SLM_WM_ATTENTION_INJECTION_PACKAGE_PATTERN', 'attention_latent_injection_package_*.zip')
os.environ.setdefault('SLM_WM_ALIGNED_RESCORING_PACKAGE_PATTERN', 'aligned_rescoring_package_*.zip')
print({
    'protocol_profile': os.environ['SLM_WM_PROTOCOL_PROFILE'],
    'prompt_set': os.environ['SLM_WM_PROMPT_SET'],
    'prompt_file': os.environ['SLM_WM_PROMPT_FILE'],
    'threshold_calibration_drive_dir': os.environ['SLM_WM_THRESHOLD_CALIBRATION_DRIVE_DIR'],
    'attention_injection_drive_dir': os.environ['SLM_WM_ATTENTION_INJECTION_DRIVE_DIR'],
    'aligned_rescoring_drive_dir': os.environ['SLM_WM_ALIGNED_RESCORING_DRIVE_DIR'],
    'target_fpr': os.environ['SLM_WM_THRESHOLD_TARGET_FPR'],
    'max_content_records': os.environ['SLM_WM_THRESHOLD_MAX_CONTENT_RECORDS'],
    'minimum_clean_negative_count': os.environ['SLM_WM_THRESHOLD_MINIMUM_CLEAN_NEGATIVE_COUNT'],
    'attention_package_pattern': os.environ['SLM_WM_ATTENTION_INJECTION_PACKAGE_PATTERN'],
    'aligned_package_pattern': os.environ['SLM_WM_ALIGNED_RESCORING_PACKAGE_PATTERN'],
})


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub


In [ ]:
import importlib.metadata as importlib_metadata
import importlib.util


def package_version_or_missing(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return 'not_installed'


dependency_report = {
    'diffusers': package_version_or_missing('diffusers'),
    'transformers': package_version_or_missing('transformers'),
    'accelerate': package_version_or_missing('accelerate'),
    'huggingface_hub': package_version_or_missing('huggingface_hub'),
    'safetensors': package_version_or_missing('safetensors'),
    'sentencepiece': package_version_or_missing('sentencepiece'),
    'protobuf': package_version_or_missing('protobuf'),
    'pillow_available': importlib.util.find_spec('PIL') is not None,
}
print(dependency_report)


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

from paper_workflow.colab_utils.threshold_calibration import run_default_threshold_calibration_from_drive_plan

threshold_calibration_summary = run_default_threshold_calibration_from_drive_plan(
    root='.',
    attention_injection_drive_dir=os.environ['SLM_WM_ATTENTION_INJECTION_DRIVE_DIR'],
    aligned_rescoring_drive_dir=os.environ['SLM_WM_ALIGNED_RESCORING_DRIVE_DIR'],
    target_fpr=float(os.environ['SLM_WM_THRESHOLD_TARGET_FPR']),
    max_content_records=os.environ['SLM_WM_THRESHOLD_MAX_CONTENT_RECORDS'],
    minimum_clean_negative_count=os.environ['SLM_WM_THRESHOLD_MINIMUM_CLEAN_NEGATIVE_COUNT'],
)
threshold_calibration_summary


In [ ]:
from pathlib import Path

summary_path = Path('outputs/threshold_calibration/threshold_calibration_result.json')
input_manifest_path = Path('outputs/threshold_calibration/threshold_calibration_input_package_manifest.json')
print(summary_path.read_text(encoding='utf-8') if summary_path.exists() else threshold_calibration_summary)
if input_manifest_path.exists():
    print(input_manifest_path.read_text(encoding='utf-8'))


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.threshold_calibration import package_threshold_calibration_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ['SLM_WM_THRESHOLD_CALIBRATION_DRIVE_DIR']
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'threshold_calibration_package_{utc_suffix}_{short_commit}.zip'
archive_record = package_threshold_calibration_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/threshold_calibration').glob('*.json')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8'))

for result_path in sorted(Path('outputs/threshold_calibration').glob('*.csv')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

assert threshold_calibration_summary['run_decision'] == 'pass', threshold_calibration_summary
assert threshold_calibration_summary['threshold_calibration_ready'] is True, threshold_calibration_summary
assert threshold_calibration_summary['geometric_rescue_ready'] is True, threshold_calibration_summary
assert archive_record.archive_digest == archive_record.drive_archive_digest, archive_record.to_dict()
